In [1]:
import nest_asyncio
nest_asyncio.apply()

import random
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

True

## 1. Simple Tools (@agent.tool_plain)

No context is passed unlike contextual tools (@agent.tool)

An agent that can roll dice.

In [9]:
game_agent = Agent(
  "groq:openai/gpt-oss-20b",
  system_prompt="You are a helpful game master assistant"
)

@game_agent.tool_plain
def roll_dice(sides: int)-> int:
  """
  Use this to roll a dice with a specific number of sides.
  
  """
  print(f"\n[System] Tool Used! Rolling a {sides}-sided dice...")
  return random.randint(1, sides)
  

In [18]:
response1 = game_agent.run_sync("I am attacking the dragon! Can you roll a 20-sided dice for me?")
print("\nAgent:", response.output)


[System] Tool Used! Rolling a 20-sided dice...

Agent: You roll a 20‑sided die and get **18**! Good luck with that mighty dragon—an 18 is a solid hit. Let me know what the dragon does next or if you need another roll.


## 2. Contextual Tools (@agent.tool)

If our tool needs to connect to a Database, we shouldn't let the LLM guess the database credentials! Instead, we should inject it using deps_type and access it via RunContext. 

In [33]:
# 1. Defining what secret data we want to pass

@dataclass
class CustomerDatabase:
  company_name: str
  secret_discounts: dict

In [34]:
# 2. Telling the Agent to expect this type of data
store_agent = Agent(
  "groq:openai/gpt-oss-20b",
  deps_type = CustomerDatabase,
  system_prompt = "You are a store assistant. Check for discounts if asked."
)

In [35]:
# 3. Access our injected data through 'ctx'
@store_agent.tool
def check_discounts(ctx: RunContext[CustomerDatabase], item_name: str)-> str:
   """Use this to check if a specific item has a discount available."""
   print(f"\n[System] Checking the {ctx.deps.company_name} DB for {item_name}...")
   
   discount = ctx.deps.secret_discounts.get(item_name.lower(), "0%")
   return f"The discount is {discount}."

In [38]:
# 4. Providing the secure data at runtime!
my_live_db = CustomerDatabase(
    company_name="Tech Mega Store",
    secret_discounts={"laptop": "20%", "mouse": "60%"}
)

In [39]:
response2 = store_agent.run_sync("Do you have any discounts on a mouse?", deps=my_live_db)
print("\nAgent:", response2.output)


[System] Checking the Tech Mega Store DB for mouse...

Agent: Yes! We’re currently offering a **60% discount** on all mice. Let me know if you’d like to place an order or need more details about the models we have in stock.
